In [11]:
import yfinance as yf
import pandas as pd
import numpy as np
from fredapi import Fred
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import time
import os
import warnings
warnings.filterwarnings('ignore')

TICKERS  = ['COST', 'WMT', 'TGT', 'LOW']
START    = '2020-01-01'
END      = '2025-12-31'

FRED_KEY = input("Write the fred key here")

os.makedirs('data/raw',       exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

print('Setup OK')
print(f'Tickers : {TICKERS}')
print(f'Period  : {START} → {END}')

Setup OK
Tickers : ['COST', 'WMT', 'TGT', 'LOW']
Period  : 2020-01-01 → 2025-12-31


In [12]:
#Daily Prices
prices_raw = yf.download(TICKERS, start=START, end=END, auto_adjust=True)['Close']
prices_raw.index = pd.to_datetime(prices_raw.index)
prices_raw.to_csv('data/raw/prices_daily.csv')

print('Shape:', prices_raw.shape)
print(prices_raw.tail(3))

[*********************100%***********************]  4 of 4 completed

Shape: (1507, 4)
Ticker            COST         LOW        TGT         WMT
Date                                                     
2025-12-26  872.158447  243.396912  98.547737  111.511147
2025-12-29  866.656067  242.739853  97.112328  112.299530
2025-12-30  864.469055  242.092743  96.449074  111.690781


In [13]:
#Getting monthly returns

monthly_close = prices_raw.resample('ME').last()

monthly_ret = monthly_close.pct_change().dropna()

monthly_ret = monthly_ret.stack().reset_index()
monthly_ret.columns = ['date', 'ticker', 'monthly_return']

monthly_ret['target'] = (monthly_ret['monthly_return'] > 0).astype(int)

monthly_ret.to_csv('data/raw/monthly_returns.csv', index=False)

print('Shape:', monthly_ret.shape)
print('\nTarget balance (should be roughly 55/45):')
print(monthly_ret['target'].value_counts(normalize=True).round(2))
print(monthly_ret.head(8))

Shape: (284, 4)

Target balance (should be roughly 55/45):
target
1    0.57
0    0.43
Name: proportion, dtype: float64
        date ticker  monthly_return  target
0 2020-02-29   COST       -0.077858       0
1 2020-02-29    LOW       -0.083190       0
2 2020-02-29    TGT       -0.064600       0
3 2020-02-29    WMT       -0.059481       0
4 2020-03-31   COST        0.014192       1
5 2020-03-31    LOW       -0.192550       0
6 2020-03-31    TGT       -0.097378       0
7 2020-03-31    WMT        0.059832       1


In [14]:
#Getting macro data from fred
fred = Fred(api_key=FRED_KEY)

vix_raw  = fred.get_series('VIXCLS', observation_start=START, observation_end=END)
rate_raw = fred.get_series('GS10',   observation_start=START, observation_end=END)

macro_raw = pd.DataFrame({'vix': vix_raw, 'rate_10y': rate_raw})
macro_raw.index = pd.to_datetime(macro_raw.index)
macro_raw.to_csv('data/raw/macro_daily.csv')

macro_monthly = macro_raw.resample('ME').mean().reset_index()
macro_monthly.columns = ['date', 'vix_avg', 'rate_10y']

macro_monthly['vix_avg']  = macro_monthly['vix_avg'].ffill()
macro_monthly['rate_10y'] = macro_monthly['rate_10y'].ffill()

# Shift values forward by 1 so each row uses the prior month's macro.
macro_monthly['vix_avg']  = macro_monthly['vix_avg'].shift(1)
macro_monthly['rate_10y'] = macro_monthly['rate_10y'].shift(1)

macro_monthly.to_csv('data/raw/macro_monthly.csv', index=False)

print('Shape:', macro_monthly.shape)
print('Missing:', macro_monthly.isnull().sum().to_dict())
print(macro_monthly.tail(4))

Shape: (72, 3)
Missing: {'date': 0, 'vix_avg': 1, 'rate_10y': 1}
         date    vix_avg  rate_10y
68 2025-09-30  15.750000      4.26
69 2025-10-31  15.789091      4.12
70 2025-11-30  18.086522      4.06
71 2025-12-31  19.769500      4.09


In [15]:
#Getting annual financials 
# Using annual statements instead of quarterly.
# Reason: yfinance quarterly data only returned 3-4 recent quarters,
# Annual data gives one value per year = 6 distinct values per feature,
# which is sufficient for EDA and hypothesis testing.

def safe_get(df, *names):
    for name in names:
        if name in df.columns:
            return pd.to_numeric(df[name], errors='coerce')
    return pd.Series(dtype=float, index=df.index)

all_funds = []

for ticker in TICKERS:
    print(f'\n── {ticker} ──')
    stock = yf.Ticker(ticker)

    try:
        inc = stock.income_stmt.T        
        inc.index = pd.to_datetime(inc.index)
        inc = inc.sort_index()
        print(f'  Annual income rows: {len(inc)}')
        print(f'  Years covered: {inc.index.year.tolist()}')
    except Exception as e:
        print(f'  Income stmt failed: {e}')
        continue

    rev   = safe_get(inc, 'Total Revenue', 'Revenue')
    op    = safe_get(inc, 'Operating Income', 'EBIT')
    gross = safe_get(inc, 'Gross Profit')
    net   = safe_get(inc, 'Net Income', 'Net Income Common Stockholders')

    op_margin    = (op / rev).replace([np.inf, -np.inf], np.nan)
    gross_margin = (gross / rev).replace([np.inf, -np.inf], np.nan)
    net_margin   = (net / rev).replace([np.inf, -np.inf], np.nan)
    op_margin_chg = op_margin.diff()

    # Annual balance sheet
    try:
        bs = stock.balance_sheet.T      
        bs.index = pd.to_datetime(bs.index)
        bs = bs.sort_index()
        print(f'  Annual balance rows: {len(bs)}')
    except Exception as e:
        print(f'  Balance sheet failed: {e}')
        bs = pd.DataFrame()

    debt      = safe_get(bs, 'Total Debt', 'Long Term Debt And Capital Lease Obligation')
    equity    = safe_get(bs, 'Stockholders Equity', 'Total Stockholders Equity', 'Common Stock Equity')
    curr_a    = safe_get(bs, 'Current Assets', 'Total Current Assets')
    curr_l    = safe_get(bs, 'Current Liabilities', 'Total Current Liabilities')

    debt      = debt.reindex(inc.index)
    equity    = equity.reindex(inc.index)
    curr_a    = curr_a.reindex(inc.index)
    curr_l    = curr_l.reindex(inc.index)

    de_ratio      = (debt / equity).replace([np.inf, -np.inf], np.nan)
    current_ratio = (curr_a / curr_l).replace([np.inf, -np.inf], np.nan)

    df = pd.DataFrame({
        'gross_margin':    gross_margin,
        'operating_margin': op_margin,
        'net_margin':       net_margin,
        'op_margin_change': op_margin_chg,
        'de_ratio':         de_ratio,
        'current_ratio':    current_ratio,
    }, index=inc.index)

    df['ticker'] = ticker

    print(f'  Rows: {len(df)}')
    print(f'  NaN counts: {df.drop(columns=["ticker"]).isnull().sum().to_dict()}')
    print(f'  Unique gross_margin values: {df["gross_margin"].nunique()}')

    df = df.reset_index().rename(columns={'index': 'date'})
    all_funds.append(df)
    time.sleep(0.5)

funds_annual = pd.concat(all_funds, ignore_index=True)
funds_annual['date'] = pd.to_datetime(funds_annual['date'])

# Keep study period
funds_annual = funds_annual[
    (funds_annual['date'] >= START) &
    (funds_annual['date'] <= END)
]

funds_annual.to_csv('data/raw/fundamentals_annual.csv', index=False)
print(f'\nTotal annual rows: {len(funds_annual)}')
print(funds_annual.groupby('ticker').size().to_string())
print(funds_annual.groupby('ticker')['gross_margin'].nunique())


── COST ──
  Annual income rows: 5
  Years covered: [2021, 2022, 2023, 2024, 2025]
  Annual balance rows: 5
  Rows: 5
  NaN counts: {'gross_margin': 1, 'operating_margin': 1, 'net_margin': 1, 'op_margin_change': 2, 'de_ratio': 1, 'current_ratio': 1}
  Unique gross_margin values: 4

── WMT ──
  Annual income rows: 5
  Years covered: [2022, 2023, 2024, 2025, 2026]
  Annual balance rows: 5
  Rows: 5
  NaN counts: {'gross_margin': 1, 'operating_margin': 1, 'net_margin': 1, 'op_margin_change': 2, 'de_ratio': 1, 'current_ratio': 1}
  Unique gross_margin values: 4

── TGT ──
  Annual income rows: 5
  Years covered: [2022, 2023, 2024, 2025, 2026]
  Annual balance rows: 5
  Rows: 5
  NaN counts: {'gross_margin': 1, 'operating_margin': 1, 'net_margin': 1, 'op_margin_change': 2, 'de_ratio': 1, 'current_ratio': 1}
  Unique gross_margin values: 4

── LOW ──
  Annual income rows: 5
  Years covered: [2022, 2023, 2024, 2025, 2026]
  Annual balance rows: 5
  Rows: 5
  NaN counts: {'gross_margin': 1, '

In [16]:
# filling quarters and adding lag

FUND_COLS = [
    'net_margin', 'gross_margin',
    'operating_margin', 'op_margin_change',
    'de_ratio', 'current_ratio'
]

monthly_index = pd.date_range(start=START, end=END, freq='ME')
filled_frames = []

for ticker, grp in funds_annual.groupby('ticker'):   #Burayı annual olarak değiştirdik
    grp = grp.set_index('date').sort_index()
    grp = grp[[c for c in FUND_COLS if c in grp.columns]]

    grp.index = grp.index + pd.DateOffset(months=3)

    combined_idx = grp.index.union(monthly_index)
    grp_exp = grp.reindex(combined_idx)

    grp_filled = grp_exp.ffill()

    grp_filled = grp_filled.bfill(limit=3)

    grp_monthly = grp_filled.reindex(monthly_index)
    grp_monthly['ticker'] = ticker
    grp_monthly = grp_monthly.reset_index().rename(columns={'index': 'date'})
    filled_frames.append(grp_monthly)

funds_monthly = pd.concat(filled_frames, ignore_index=True)

# median imputation per ticker for anything still missing
for col in FUND_COLS:
    if col in funds_monthly.columns:
        before = funds_monthly[col].isnull().sum()
        funds_monthly[col] = funds_monthly.groupby('ticker')[col].transform(
            lambda x: x.fillna(x.median())
        )
        after = funds_monthly[col].isnull().sum()
        if before > 0:
            print(f'  {col}: {before} NaN → {after} after median imputation')

funds_monthly.to_csv('data/raw/fundamentals_monthly.csv', index=False)

print('\nFinal NaN % per column:')
print(funds_monthly[FUND_COLS].isnull().mean().mul(100).round(1).to_string())
print(f'\nShape: {funds_monthly.shape}')
print(funds_monthly.head(6))

  net_margin: 139 NaN → 0 after median imputation
  gross_margin: 139 NaN → 0 after median imputation
  operating_margin: 139 NaN → 0 after median imputation
  op_margin_change: 187 NaN → 0 after median imputation
  de_ratio: 139 NaN → 0 after median imputation
  current_ratio: 139 NaN → 0 after median imputation

Final NaN % per column:
net_margin          0.0
gross_margin        0.0
operating_margin    0.0
op_margin_change    0.0
de_ratio            0.0
current_ratio       0.0

Shape: (288, 8)
        date  net_margin  gross_margin  operating_margin  op_margin_change  \
0 2020-01-31    0.025969      0.122597          0.034337         -0.000849   
1 2020-02-29    0.025969      0.122597          0.034337         -0.000849   
2 2020-03-31    0.025969      0.122597          0.034337         -0.000849   
3 2020-04-30    0.025969      0.122597          0.034337         -0.000849   
4 2020-05-31    0.025969      0.122597          0.034337         -0.000849   
5 2020-06-30    0.025969      0

In [17]:
# sentiment + vader

analyzer = SentimentIntensityAnalyzer()

demo = [
    ('COST', 'Costco reports record quarterly profit, beats analyst estimates'),
    ('TGT',  'Target slashes guidance as inventory glut crushes margins'),
    ('WMT',  'Walmart raises full-year outlook on resilient grocery demand'),
    ('LOW',  "Lowe's misses revenue estimates as housing market cools sharply"),
]

print('VADER demo (compound score: -1 = very negative, +1 = very positive):')
print(f'{"Ticker":<8} {"Score":>7}   Headline')
print('-' * 70)
for ticker, headline in demo:
    score = analyzer.polarity_scores(headline)['compound']
    print(f'{ticker:<8} {score:>+7.3f}   {headline}')

sent_monthly = pd.DataFrame(columns=['ticker', 'date', 'sentiment_avg'])
sent_monthly.to_csv('data/raw/sentiment_monthly.csv', index=False)
print('\nPlaceholder saved — sentiment_avg will be 0.0 in master dataset.')

VADER demo (compound score: -1 = very negative, +1 = very positive):
Ticker     Score   Headline
----------------------------------------------------------------------
COST      +0.440   Costco reports record quarterly profit, beats analyst estimates
TGT       -0.572   Target slashes guidance as inventory glut crushes margins
WMT       -0.128   Walmart raises full-year outlook on resilient grocery demand
LOW       -0.226   Lowe's misses revenue estimates as housing market cools sharply

Placeholder saved — sentiment_avg will be 0.0 in master dataset.


In [18]:
# reload centiment (because of having problems in sentiments, more work will be done in next steps)
sent_monthly = pd.read_csv('data/raw/sentiment_monthly.csv')
print(f'Sentiment rows loaded: {len(sent_monthly)}')
print('sentiment_avg will be set to 0.0 during merge.')

Sentiment rows loaded: 0
sentiment_avg will be set to 0.0 during merge.


In [19]:
# Merging data

# Start with returns as the spine
master = monthly_ret.copy()
master['date'] = pd.to_datetime(master['date'])

# 1. Macro (lagged)
macro_monthly['date'] = pd.to_datetime(macro_monthly['date'])
master = master.merge(macro_monthly, on='date', how='left')
print(f'After macro merge       : {master.shape}')

# 2. Fundamentals (lagged)
funds_monthly['date'] = pd.to_datetime(funds_monthly['date'])
master = master.merge(funds_monthly, on=['date', 'ticker'], how='left')
print(f'After fundamentals merge: {master.shape}')

# 3. Sentiment, fill all NaN with 0.0 (neutral)
if len(sent_monthly) > 0:
    sent_monthly['date'] = pd.to_datetime(sent_monthly['date'])
    master = master.merge(sent_monthly, on=['date', 'ticker'], how='left')
else:
    master['sentiment_avg'] = 0.0

master['sentiment_avg'] = master['sentiment_avg'].fillna(0.0)

# Clean column order
col_order = [
    'date', 'ticker',
    'monthly_return', 'target',
    'vix_avg', 'rate_10y',
    'net_margin', 'gross_margin',
    'operating_margin', 'op_margin_change',
    'de_ratio', 'current_ratio',
    'sentiment_avg'
]
master = master[[c for c in col_order if c in master.columns]]
master = master.sort_values(['ticker', 'date']).reset_index(drop=True)

master.to_csv('data/processed/master_dataset.csv', index=False)
print(f'\nSaved → data/processed/master_dataset.csv')
print(f'Shape: {master.shape}')
print(master.head(8))

After macro merge       : (284, 6)
After fundamentals merge: (284, 12)

Saved → data/processed/master_dataset.csv
Shape: (284, 13)
        date ticker  monthly_return  target    vix_avg  rate_10y  net_margin  \
0 2020-02-29   COST       -0.077858       0  13.940952      1.76    0.025969   
1 2020-03-31   COST        0.014192       1  19.628947      1.50    0.025969   
2 2020-04-30   COST        0.065101       1  57.736818      0.87    0.025969   
3 2020-05-31   COST        0.018052       1  41.453810      0.66    0.025969   
4 2020-06-30   COST       -0.017052       0  30.897000      0.67    0.025969   
5 2020-07-31   COST        0.075922       1  31.119545      0.73    0.025969   
6 2020-08-31   COST        0.067981       1  26.840455      0.62    0.025969   
7 2020-09-30   COST        0.021112       1  22.889524      0.65    0.025969   

   gross_margin  operating_margin  op_margin_change  de_ratio  current_ratio  \
0      0.122597          0.034337         -0.000849  0.354537       

In [20]:
# validation of results

print('=' * 55)
print('  VALIDATION CHECKLIST')
print('=' * 55)

checks_passed = 0

# shape
rows, cols = master.shape
shape_ok = (240 <= rows <= 310) and (cols >= 11)
print(f'\n[1] Shape: {master.shape}')
print(f'    Expect ~288 rows, 13 cols  →  {"OK" if shape_ok else "CHECK"}')
if shape_ok: checks_passed += 1

# rows per ticker
ticker_counts = master.groupby('ticker').size()
balanced = ticker_counts.std() < 3
print(f'\n[2] Rows per ticker:')
print('    ' + ticker_counts.to_string().replace('\n', '\n    '))
print(f'    All roughly equal?  →  {"OK" if balanced else "CHECK"}')
if balanced: checks_passed += 1

# date ranger
date_ok = (str(master['date'].min())[:7] == '2020-01') and \
          (str(master['date'].max())[:7] == '2025-12')
print(f'\n[3] Date range: {master["date"].min().date()} → {master["date"].max().date()}')
print(f'    Expect 2020-01-31 → 2025-12-31  →  {"OK" if date_ok else "CHECK"}')
if date_ok: checks_passed += 1

# target balance
balance = master['target'].value_counts(normalize=True)
balance_ok = 0.35 <= balance.min() <= 0.65
print(f'\n[4] Target balance:')
print(f'    Up months   (1): {balance.get(1, 0):.0%}')
print(f'    Down months (0): {balance.get(0, 0):.0%}')
print(f'    Not severely imbalanced?  →  {"OK" if balance_ok else "CHECK"}')
if balance_ok: checks_passed += 1

# missing values
total_nan = master.isnull().sum().sum()
nan_ok = total_nan == 0
print(f'\n[5] Missing values per column:')
print('    ' + master.isnull().sum().to_string().replace('\n', '\n    '))
print(f'    Total NaN: {total_nan}  →  {"OK" if nan_ok else "CHECK"}')
if nan_ok: checks_passed += 1

# summary
print(f'\n{"=" * 55}')
print(f'  Result: {checks_passed}/5 checks passed')
if checks_passed == 5:
    print('  Day 1 COMPLETE — master_dataset.csv ready for EDA')
else:
    print('  Share this output and we will fix remaining issues')
print('=' * 55)

# one sample row from the result
print('\nSample row — COST, first available month:')
print(master[master['ticker'] == 'COST'].iloc[0].to_string())

  VALIDATION CHECKLIST

[1] Shape: (284, 13)
    Expect ~288 rows, 13 cols  →  OK

[2] Rows per ticker:
    ticker
    COST    71
    LOW     71
    TGT     71
    WMT     71
    All roughly equal?  →  OK

[3] Date range: 2020-02-29 → 2025-12-31
    Expect 2020-01-31 → 2025-12-31  →  CHECK

[4] Target balance:
    Up months   (1): 57%
    Down months (0): 43%
    Not severely imbalanced?  →  OK

[5] Missing values per column:
    date                0
    ticker              0
    monthly_return      0
    target              0
    vix_avg             0
    rate_10y            0
    net_margin          0
    gross_margin        0
    operating_margin    0
    op_margin_change    0
    de_ratio            0
    current_ratio       0
    sentiment_avg       0
    Total NaN: 0  →  OK

  Result: 4/5 checks passed
  Share this output and we will fix remaining issues

Sample row — COST, first available month:
date                2020-02-29 00:00:00
ticker                             COST
mon